## RAG린 무엇인가?
LLM이 답변하기 전에 외부 문서를 검색하고, 검색 결과를 LLM의 입력으로 함께 전달하는 방식입니다.

## RAG 처리 과정

1. PDF 파일에서 텍스트와 페이지 정보를 읽습니다.
2. 긴 텍스트를 청크로 나눕니다.
3. 각 청크를 임베딩으로 변환해 벡터 저장소에 저장합니다.
4. 사용자가 질문하면 질문도 임베딩으로 변환합니다.
5. 저장소에서 질문과 관련된 청크를 몇 개 찾습니다.
6. 질문과 찾은 청크를 LLM에 전달해 답변을 생성합니다.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader_1 = PyPDFLoader('data/OneNYC_2050_Strategic_Plan.pdf')
data_pdf_1 = loader_1.load()

loader_2 = PyPDFLoader('data/2040_seoul_plan.pdf')
data_pdf_2 = loader_2.load()

C:\Users\Unreon\AppData\Local\Temp\ipykernel_18152\1252779125.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/SLGUZI+DINNextLTPro-Bold', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(5397, 0, 2834508391024), '/LastChar': 110, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [236, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 556, 556, 0, 0, 556, 0, 0, 0, 556, 0, 0, 0, 0, 0, 0, 0, 644, 610, 575, 630, 565, 557, 628, 658, 277, 0, 645, 550, 763, 655, 628, 605, 628, 624, 595, 557, 645, 595, 865, 595, 568, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 527, 0, 0, 0, 0, 0, 0, 0, 0, 553]}, but is not insta

## 1. PDF를 읽어 Document 목록으로 변환

이 셀은 PDF 파일 두 개를 읽습니다. 하나는 뉴욕 계획 문서이고, 다른 하나는 서울 계획 문서입니다. PDF를 읽은 결과는 문자열 하나가 아니라 Document 객체가 여러 개 들어 있는 목록입니다. 보통 PDF의 페이지별 텍스트가 여러 Document로 만들어집니다.

Document에는 다음 두 가지 정보가 있습니다.

- **page_content**: PDF에서 읽어 낸 본문 텍스트입니다. 이후 단계는 이 텍스트를 나누고 검색합니다.
- **metadata**: 파일 이름, 페이지 번호처럼 이 텍스트가 어디에서 왔는지 알려 주는 정보입니다.

나중에 답변의 출처를 표시하려면 metadata가 필요합니다. 예를 들어 답변에 사용한 PDF 파일과 페이지 번호를 함께 보여 줄 수 있습니다.
PDF 파일을 읽는 데 성공했다고 해서 텍스트 품질까지 보장되는 것은 아닙니다. PDF에는 표, 두 개 이상으로 나뉜 문단, 각주, 이미지 안의 글자가 포함될 수 있습니다.  
이런 부분은 읽은 텍스트의 순서가 달라지거나 누락될 수 있습니다. 이미지로만 이루어진 PDF는 별도의 문자 인식 과정이 필요할 수 있습니다.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits_1 = text_splitter.split_documents(data_pdf_1)
splits_2 = text_splitter.split_documents(data_pdf_2)

## 2. 청킹: 검색의 최소 단위를 설계

RAG는 PDF 전체가 아니라 질문과 관련된 텍스트 일부를 찾아 LLM에 전달합니다. 이때 긴 Document를 작은 단위로 나누는 작업이 필요합니다.

### 2.1. 청크

청크란 긴 Document를 나눈 작은 텍스트 조각입니다. 검색과 LLM 입력은 청크 단위로 처리됩니다.

**왜 나누는가?**
- PDF 전체를 사용하면 질문과 관계없는 내용이 함께 포함될 수 있습니다.
- LLM에 전달하는 텍스트가 너무 길어질 수 있습니다.
- 질문에 필요한 부분만 선택하기 어렵습니다.

**현재 설정**

| 설정 | 의미 |
| --- | --- |
| `chunk_size` | 1000 | 청크 하나에 넣을 최대 문자 수 |

`RecursiveCharacterTextSplitter`는 빈 줄과 줄바꿈을 먼저 기준으로 사용합니다. 청크가 여전히 길면 공백이나 문자 위치에서 나눕니다.

### 2.2. 오버랩

오버랩이란 인접한 청크에 공통으로 포함되는 텍스트입니다. 청크 경계에서 문장이 나뉘는 문제를 줄이기 위해 사용합니다.

**현재 설정**

| 설정 | 값 | 의미 |
| --- | --- |
| `chunk_overlap` | 청크 끝부분의 100자를 다음 청크에도 포함 |

### 2.3. 적절한 청크와 오버랩의 값은 어떻게 알 수 있는가?
청크와 오버랩에는 모든 문서에 적용되는 정답이 없습니다. PDF의 구조, 사용자가 묻는 질문, 검색 결과를 기준으로 값을 정합니다.

**시작 값**
현재 노트북의 설정인 `chunk_size=1000`, `chunk_overlap=100`을 시작 값으로 사용할 수 있습니다. 이 값이 항상 적절하다는 뜻은 아니며, 검색 결과를 확인한 뒤 조정합니다.

**확인 방법**
1. 실제로 사용할 질문을 5개 정도 준비합니다.
2. 각 질문으로 문서를 검색합니다.
3. 검색된 청크에 답변에 필요한 문장이 포함되어 있는지 확인합니다.
4. 한 번에 하나의 값만 바꾼 뒤, 같은 질문으로 다시 검색합니다.

**검색 결과에 따른 조정**
| 검색 결과 | 조정 방법 |
| --- | --- |
| 필요한 문장이 청크 두 개 이상으로 나뉨 | `chunk_size` 또는 `chunk_overlap`을 늘립니다. |
| 한 청크에 관계없는 내용이 너무 많음 | `chunk_size`를 줄입니다. |
| 비슷한 청크가 여러 개 반복됨 | `chunk_overlap`을 줄입니다. |
| 제목과 본문, 수치와 조건이 분리됨 | `chunk_size`를 늘리거나 문단 단위 분할을 확인합니다. |

청크와 오버랩 값을 바꾼 뒤에는 기존 벡터 저장소를 다시 만들어야 합니다. 기존 저장소에는 이전 설정으로 나눈 청크가 이미 저장되어 있기 때문입니다.

In [3]:
print(type(splits_1[0]))
print(type(splits_2[0]))

<class 'langchain_core.documents.base.Document'>
<class 'langchain_core.documents.base.Document'>


In [4]:
print(splits_2[50].page_content)
print('----------------------')
print(splits_2[51].page_content)

for i in range(len(splits_2) - 1):
    splits_2[i].page_content += "\n"+ splits_2[i + 1].page_content[:100]

print(splits_2[50].page_content)
print('----------------------')
print(splits_2[51].page_content)

34 제2장 미래상과 목표
3. 서울의 미래 여건 변화와 과제
1) 가속화되는 저출생·고령화
2040년 서울의 고령인구는 약 32%, 늘어나는 복지·의료 
수요 대응 필요
Ÿ 장기적인 저출생·고령화로 인해 인구변화 속도는 둔화되는 
반면, 기술발전에 따른 평균수명 증가로 고령인구의 비율은 
지속적으로 상승 중이다. 
Ÿ 서울의 고령화 속도는 빠른 편으로 노년인구 비율은 2018년 
기준 14.4%로 고령사회에 진입하였으며, 2026년에는 초
고령화사회에 진입할 것으로 예측된다.
2040년 서울도시기본계획 계획인구는 통계청 전망을 반영한 854만 명으로 설정
Ÿ 2040 서울도시기본계획의 계획인구는 장기적으로 발생하고 있는 인구 감소와 고령화, 저성장 등 도시
환경 변화를 반영하는 지표로 도시의 환경용량, 인프라 수요 등을 결정
Ÿ 2040년 계획인구는 다음의 근거를 고려하여 통계청의 추계인구(중위)인 854만 명으로 설정
- 첫째, 광역 교통망의 발달로 수도권은 이미 하나의 생활권이 되었으며, 향후 수도권 광역급행철도 등 
신규 교통수단의 발달로 광역 생활권의 범위 확대는 더욱 가속화될 것으로 전망
- 둘째, 서울의 인구는 감소할 것으로 전망되지만 서울로 오가며 생활하는 인구는 오히려 늘어날 것으로 
전망되어 거주 인구로만은 판단하기 어려운 복잡한 인구이동 특성을 보유
Ÿ 서울 대도시권으로의 공간적 확장, 인접지역 간 발생하는 생활인구 등 복잡한 현황을 고려하여 인구감소 
추세에 대응하고 시민 삶의 질 향상을 위한 계획인구를 설정
2) 일상의 반강제적 디지털 전환에 대응
디지털 전환에 따른 시민 일상생활 및 도시공간 변화 감지
Ÿ 2019년 갑작스럽게 발생한 코로나19는 도시의 산업구조뿐만 아니라 일상에서 반강제적 디지털 전환
을 가속화하였다.
- 원격의료, 재택근무, 온라인 교육 등 다양한 온라인 기반 플랫폼의 등장
- 중급역량4)이 2021년 기준, 2019년 대비 10.1% 증가한 74.1%로 높은 증가 추세
----------------------
- 

In [5]:
all_splits = splits_1
print(len(all_splits))

all_splits.extend(splits_2)
print(len(all_splits))

1023
1312


In [6]:
from langchain_openai import OpenAIEmbeddings 
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
v = embedding.embed_query("뉴욕의 온실가스 저감 정책은 뭐야?")
print(v)
print(len(v))

[-0.00536346435546875, -0.049530029296875, -0.01250457763671875, 0.00847625732421875, -0.040069580078125, -0.0104217529296875, 0.0063018798828125, 0.025421142578125, -0.049774169921875, -0.0123291015625, -0.0019893646240234375, -0.017822265625, 0.026275634765625, -0.0010824203491210938, 0.00240325927734375, -0.0182342529296875, -0.0389404296875, -0.043182373046875, 0.07275390625, -0.012054443359375, 0.0494384765625, -0.0147857666015625, -0.0191192626953125, 0.0059967041015625, -0.0328369140625, 0.0169830322265625, 0.0250396728515625, 0.02020263671875, -0.004241943359375, 0.037750244140625, 0.035736083984375, 0.03057861328125, 0.0175628662109375, -0.00931549072265625, -0.0178985595703125, 0.01371002197265625, 0.06256103515625, -0.03033447265625, 0.04901123046875, -0.00263214111328125, -0.0274200439453125, 0.0129241943359375, -0.032989501953125, -0.00745391845703125, 0.0195770263671875, -0.0031528472900390625, 0.01071929931640625, -0.03094482421875, -0.0082550048828125, -0.00547790527343

In [7]:
from langchain_chroma import Chroma
import os

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )

Loading existing Chroma store


## 3. 임베딩과 벡터 저장소: 텍스트를 검색 가능한 데이터로 저장

### 3.1. 임베딩
임베딩은 텍스트를 여러 숫자로 이루어진 목록으로 변환하는 작업입니다. 이 숫자 목록은 사람이 읽기 위한 값이 아니라, 질문과 문서가 얼마나 관련 있는지 계산하기 위해 사용합니다.  

| 코드 요소 | 역할 |
| --- | --- |
| `OpenAIEmbeddings` | 텍스트를 임베딩으로 변환하는 도구 |
| `embed_query` | 질문 한 문장을 임베딩으로 변환하는 함수 |
| `text-embedding-3-large` | 현재 사용하는 임베딩 모델 |

출력되는 숫자와 숫자 개수를 이해할 필요는 없습니다. 이 출력은 임베딩이 정상적으로 생성되었는지 확인하는 용도입니다.  

**주의할 점**  
문서 청크와 사용자 질문은 같은 임베딩 모델로 변환해야 합니다. 서로 다른 모델을 사용하면 숫자 목록을 비교한 결과를 신뢰하기 어렵습니다.

### 3.2. 벡터 저장소
벡터 저장소는 임베딩과 원래 텍스트를 함께 저장하는 데이터 저장소입니다. 사용자가 질문하면 질문의 임베딩과 관련된 문서 청크를 찾아 반환합니다.
이 노트북에서는 Chroma를 벡터 저장소로 사용합니다.

| 저장되는 정보 | 사용 목적 |
| --- | --- |
| 임베딩 | 질문과 관련된 청크를 찾기 위해 사용 |
| 청크 텍스트 | LLM이 답변을 만들 때 참고 |
| 파일 이름과 페이지 정보 | 검색 결과의 출처를 확인할 때 사용 |

### 3.3. 저장소를 다시 만들어야 하는 경우
현재 코드는 `chroma_store` 폴더가 없으면 새 저장소를 만들고, 폴더가 있으면 기존 저장소를 불러옵니다.
다음 내용을 변경했다면 기존 저장소를 삭제하고 다시 만들어야 합니다.

- PDF 파일을 추가하거나 교체한 경우
- `chunk_size` 또는 `chunk_overlap` 값을 바꾼 경우
- 임베딩 모델을 바꾼 경우

기존 저장소에는 이전 PDF와 이전 청크 설정으로 만든 데이터가 남아 있기 때문입니다.

### 3.4. 임베딩 검색의 제한

임베딩 검색은 질문과 관련된 텍스트를 찾는 기능입니다. 다음 내용까지 자동으로 판단하지는 않습니다.

- 문서 내용이 사실인지
- 문서가 최신인지
- 검색된 청크만으로 질문에 충분히 답할 수 있는지

따라서 검색 결과와 원문 PDF를 함께 확인해야 합니다.

In [8]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke("서울시의 환경 정책에 대해 궁금해")

for d in docs:
    print(d)
    print('------')

page_content='78 제3장 부문별 전략계획
제4절 
기후·환경 부문
1. 개요
Ÿ 기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측
면과 연관되어 있어 향후 서울시의 적극적인 대응이 필요하다.
Ÿ 탄소중립 목표뿐만 아니라 미세먼지로부터 시민 건강을 지키기 위해서는 건물, 교통, 에너지 등 
도시의 주요 인프라 전반의 혁신이 요구되며, 이를 위해 새로운 기술과 혁신적 제도가 필요하다. 
제로에너지 건물, 친환경 차량 및 교통 인프라의 확대, 자원·에너지 순환 기반 조성으로 온실가스와 
미세먼지 배출량을 획기적으로 감축해야 한다. 
Ÿ 기후변화에 따른 폭염, 풍수해, 도심열섬현상 등 기후재난 및 극한 기후현상이 심해질 것으로 전망
되어 보다 능동적인 대비가 필요하다. 
Ÿ 한편, 환경보존과 쾌적한 도시환경을 위해 도심 곳곳 시민 모두가 누릴 수 있는 도심숲과 생활공원 
등 녹색공간을 조성하고, 이를 수변 공간과 연계하여 풍부하고 지속가능한 자연환경이 확보될 수 
있도록 한다.
Ÿ 장기적인 측면에서 시민 개개인과 기업 등 다양한 도시 내 행위자의 적극적인 협조가 필수적이며 
이를 위해 중앙정부와 서울시 환경계획 담당부서와의 협력적이고 포용적인 거버넌스 체계를 구축
하도록 한다.
목표 전략
3-1 2050 탄소중립 실현을 위한 
도시 인프라 전환
3-1-1건물 부문의 탄소배출을 감축하기 위한 친환경 기술 개발 및 적극 적용
3-1-2미래 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라 확충
3-1-3에너지 전환을 위한 청정에너지 기반 구축
3-1-4대기 환경을 고려한 공간계획과 배출원 관리체계 강화
3-2 건강한 순환도시 조성을 위한
자립적인 자원순환 체계 구축
3-2-1자원순환·관리 자립을 위한 분산형 폐기물처리 시설 구축
3-2-2기후 행동 포용적 거버넌스 구축을 위한 시민 행동 활성화
3-3 사람과 자연의 공존을 위한
친환경 생태도시 구축
3-3-1건물 에너지 분야 효율성 개선 및 도심 속 생물

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-5.6-luna")

# ④
question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        ( # ⑤
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

## 4. 관련 청크를 찾고 LLM에 전달

### 4.1. Retriever와 검색 결과
retriever는 사용자 질문을 입력받아 관련된 Document 청크 목록을 반환합니다. 이 목록은 최종 답변이 아니라, LLM이 답변을 만들 때 참고할 텍스트입니다.
현재 코드에서 `docs` 변수에는 검색된 청크 목록이 저장됩니다. `docs`를 출력하면 실제로 어떤 텍스트가 LLM에 전달될지 확인할 수 있습니다.

**검색 결과에서 확인할 점**
- 질문과 같은 도시, 정책, 기간을 다루는가?
- 답변에 필요한 수치나 조건이 포함되어 있는가?
- 같은 내용이 여러 청크에 반복되지 않는가?
- 원문에 답이 있는데 검색 결과에는 없는가?

검색 결과에 필요한 내용이 없다면, LLM이 정확한 답변을 만들기 어렵습니다. 이 경우에는 프롬프트보다 문서, 청크, 임베딩, 검색 설정을 먼저 확인합니다.

### 4.2. 검색할 청크 개수: `k`

`k`는 질문마다 가져올 청크 개수입니다.

| `k` 값 | 결과 |
| --- | --- |
| 너무 작음 | 필요한 청크가 검색 결과에서 빠질 수 있음 |
| 너무 큼 | 관계없는 내용이나 중복된 내용도 LLM에 전달될 수 있음 |

처음에는 작은 값으로 시작하고, `docs` 출력 결과를 확인하면서 조정합니다. 질문에 필요한 내용이 자주 빠지면 값을 늘리고, 비슷한 청크가 반복되면 값을 줄입니다.

### 4.3. 컨텍스트 만들기

`create_stuff_documents_chain`은 검색된 여러 청크를 하나의 텍스트로 합쳐 LLM에 전달합니다. 이 텍스트를 **컨텍스트**라고 합니다.

LLM은 다음 정보를 함께 입력받습니다.

| 입력 정보 | 용도 |
| --- | --- |
| 사용자 질문 | 사용자가 알고 싶은 내용 |
| 이전 대화 메시지 | 짧은 후속 질문의 의미 확인 |
| 컨텍스트 | 검색된 문서의 내용 |

LLM은 컨텍스트를 참고해 답변을 생성합니다.

### 4.4. 프롬프트에 포함할 규칙

현재 프롬프트는 컨텍스트를 바탕으로 답하라고 지시합니다. 답변의 신뢰도를 높이려면 다음 규칙을 추가할 수 있습니다.

1. 컨텍스트에 없는 내용은 사실처럼 답하지 않습니다.
2. 컨텍스트에 답이 없으면 찾을 수 없다고 답합니다.
3. 가능하면 PDF 파일 이름과 페이지 번호를 답변에 표시합니다.

### 4.5. RAG가 자동으로 확인하지 않는 내용

RAG는 질문과 관련된 텍스트를 찾고 LLM에 전달합니다. 하지만 다음 내용은 자동으로 판단하지 않습니다.

- 검색된 문서가 최신인지
- 문서 내용이 사실인지
- 문서 내용만으로 질문에 충분히 답할 수 있는지

따라서 검색 결과와 원문 PDF를 함께 확인해야 합니다.

In [13]:
from langchain_core.chat_history import (
    InMemoryChatMessageHistory as ChatMessageHistory,
)

# 채팅 메시지 저장할 메모리 객체 생성
chat_history = ChatMessageHistory() 
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("서울시의 온실가스 저감 정책에 대해 알려줘.") 

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

서울시는 **2050년 탄소중립 실현**을 목표로 건물·교통·에너지·폐기물·도시계획 전반의 전환을 추진하고 있습니다. 주요 정책은 다음과 같습니다.

### 1. 건물 부문 탄소배출 감축
- 제로에너지건축물 등 **친환경 건축기술의 개발·적용**
- 건물의 에너지 효율 개선과 에너지 절약형 설비 확대
- 기존 건축물의 에너지 성능 개선
- 건물 에너지 분야의 효율성을 높이고, 온실가스 배출을 줄이는 관리체계 강화

### 2. 친환경 교통·미래 모빌리티 확대
- 전기차·수소차 등 **친환경 수송 차량 보급 확대**
- 충전시설 등 관련 교통 인프라 확충
- 미래 모빌리티 기술을 활용해 교통 부문의 화석연료 사용과 배출가스 감축
- 친환경 교통체계 전환을 통해 온실가스와 미세먼지를 함께 저감

### 3. 청정에너지 기반 구축
- 태양광 등 **소규모 분산형 발전시설 확대**
- 공공시설의 옥상과 주차장 등을 활용한 지역 에너지 생산
- 지역에서 생산한 전력을 지역 내에서 활용하고, 잉여 전력을 거래할 수 있는 제도 마련
- 에너지 생산·관리의 지역화를 통해 청정에너지와 순환형 에너지 경제 기반 조성

### 4. 대기환경을 고려한 도시계획과 배출원 관리
- 도시의 대기환경 수용능력과 자연적인 대기순환을 고려한 개발·정비계획 수립
- 건축물 배치와 바람길 확보를 고려한 도시공간 관리
- 배출원별·계절별 특성을 반영해 PM2.5, 질소산화물(NOx) 등 대기오염물질을 원천적으로 감축
- 온실가스 감축과 대기질 개선을 연계한 맞춤형 관리 추진

### 5. 자원순환과 폐기물 감량
- 생활권·자치구·광역권 단위의 **분산형 자원순환 및 폐기물처리시설 확충**
- 폐기물의 원천 감량, 재사용, 재활용, 새활용 확대
- 재활용 신기술 개발과 재활용 산업의 성장 지원
- 사업장과 생활폐기물의 감량 및 온실가스 저감을 통합 관리
- 5대 권역별 자원순환 클러스터를 조성해 폐기물 처리의 지역 자립성 강화

### 6. 시민·기업·정부가 참여하는 기후 거버넌스
- 서울시, 중앙

In [14]:
for m in chat_history.messages:
    print(m)

content='서울시의 온실가스 저감 정책에 대해 알려줘.' additional_kwargs={} response_metadata={}
content='서울시는 **2050년 탄소중립 실현**을 목표로 건물·교통·에너지·폐기물·도시계획 전반의 전환을 추진하고 있습니다. 주요 정책은 다음과 같습니다.\n\n### 1. 건물 부문 탄소배출 감축\n- 제로에너지건축물 등 **친환경 건축기술의 개발·적용**\n- 건물의 에너지 효율 개선과 에너지 절약형 설비 확대\n- 기존 건축물의 에너지 성능 개선\n- 건물 에너지 분야의 효율성을 높이고, 온실가스 배출을 줄이는 관리체계 강화\n\n### 2. 친환경 교통·미래 모빌리티 확대\n- 전기차·수소차 등 **친환경 수송 차량 보급 확대**\n- 충전시설 등 관련 교통 인프라 확충\n- 미래 모빌리티 기술을 활용해 교통 부문의 화석연료 사용과 배출가스 감축\n- 친환경 교통체계 전환을 통해 온실가스와 미세먼지를 함께 저감\n\n### 3. 청정에너지 기반 구축\n- 태양광 등 **소규모 분산형 발전시설 확대**\n- 공공시설의 옥상과 주차장 등을 활용한 지역 에너지 생산\n- 지역에서 생산한 전력을 지역 내에서 활용하고, 잉여 전력을 거래할 수 있는 제도 마련\n- 에너지 생산·관리의 지역화를 통해 청정에너지와 순환형 에너지 경제 기반 조성\n\n### 4. 대기환경을 고려한 도시계획과 배출원 관리\n- 도시의 대기환경 수용능력과 자연적인 대기순환을 고려한 개발·정비계획 수립\n- 건축물 배치와 바람길 확보를 고려한 도시공간 관리\n- 배출원별·계절별 특성을 반영해 PM2.5, 질소산화물(NOx) 등 대기오염물질을 원천적으로 감축\n- 온실가스 감축과 대기질 개선을 연계한 맞춤형 관리 추진\n\n### 5. 자원순환과 폐기물 감량\n- 생활권·자치구·광역권 단위의 **분산형 자원순환 및 폐기물처리시설 확충**\n- 폐기물의 원천 감량, 재사용, 재활용, 새활용 확대\n- 재활용 신기술 개발과 재활용 산업의 성장 지원\n

In [15]:
from langchain_core.output_parsers import StrOutputParser # 문자열 출력 파서를 불러옵니다.

query_for_nyc = "뉴욕은?"

# query augmentation 
# 기존 대화 내용을 활용해 query_augmentation 수행
query_augmentation_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="messages"), # 기존 대화 내용
        (
            "system",
            "기존의 대화 내용을 활용하여 사용자의 아래 질문의 의도를 파악하여 명료한 한 문장의 질문으로 변환하라. 대명사나 이, 저, 그와 같은 표현을 명확한 명사로 표현하라. :\n\n{query}",
        ),
    ]
)

query_augmentation_chain = query_augmentation_prompt | chat | StrOutputParser()


augmented_query = query_augmentation_chain.invoke({
    "messages": chat_history.messages,
    "query": query_for_nyc  
})

print(augmented_query)

docs = retriever.invoke(augmented_query)

for d in docs:
    print(d)
    print('------')

뉴욕시의 온실가스 저감 정책은 무엇인가?
page_content='New York City, because of its density and public transportation 
system, has long had a smaller per capita carbon footprint than 
any other big city in the United States — and we have made 
significant progress reducing GHG emissions over the last decade, 
using new technologies and innovations to get us there. The City 
has assumed a leading global role in fighting climate change, 
and the actions we take can become a national and global model. 
However, the lack of commitment by the federal government 
to the Paris Agreement has placed New York and the world in a 
precarious position. Time is running out.
While New York City has made strides to achieve a reduction in 
greenhouse gas emissions, global emissions continue to rise, 
putting New Yorkers at risk. 
CHANGE IN GHG EMISSIONS, 2005-2017
Source: Mayor’s Office, International Energy Agency
0%
5%
GLOBAL
NEW YORK CITY-5%
15%
-15%
-20%
2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2

## 5. 대화형 RAG: 생략된 질문을 검색 가능한 질문으로 변환

### 5.1. 짧은 후속 질문의 문제

대화가 이어지면 사용자는 앞에서 말한 내용을 생략할 수 있습니다.

예를 들어 앞 대화가 서울시의 온실가스 저감 정책에 대한 내용이었다면, `뉴욕은?`이라는 질문은 뉴욕시의 온실가스 저감 정책을 묻는 것일 수 있습니다. 하지만 질문 문장만으로는 어떤 주제와 조건을 유지해야 하는지 충분히 알 수 없습니다.

검색 기능은 질문 문장으로 관련 청크를 찾으므로, 짧은 질문을 더 명확한 질문으로 바꾸는 과정이 필요합니다.

### 5.2. 질문 재작성

이 노트북의 query augmentation 코드는 이전 대화 메시지와 새 질문을 LLM에 전달합니다. LLM은 짧거나 생략된 질문을 검색에 사용할 수 있는 한 문장으로 바꿉니다.

| 구분 | 예시 |
| --- | --- |
| 원래 질문 | 뉴욕은? |
| 재작성된 질문 | 뉴욕시의 온실가스 저감 정책은 무엇인가? |

재작성된 질문은 최종 답변이 아닙니다. 문서를 검색하기 위한 입력값입니다.

### 5.3. 대화 이력과 PDF의 역할

| 정보 | 사용 목적 |
| --- | --- |
| 대화 이력 | 사용자가 무엇을 가리키는지 이해 |
| 검색된 PDF | 답변에 포함할 사실과 출처 확인 |

이전 대화에 포함된 LLM 답변은 틀릴 수 있습니다. 따라서 새로운 사실을 답변할 때는 재작성된 질문으로 PDF를 다시 검색해야 합니다.

### 5.4. 재작성된 질문 확인하기

재작성된 질문을 출력하면 검색 결과가 예상과 다른 이유를 확인할 수 있습니다.

다음 내용을 확인합니다.

- 원래 질문의 도시, 기간, 조건이 유지되었는가?
- 앞 대화와 관계없는 내용이 추가되지 않았는가?
- 재작성된 질문이 사용자의 의도와 같은가?

### 5.5. 대화 이력의 저장 범위

현재 `ChatMessageHistory`는 노트북이 실행되는 동안에만 메시지를 저장합니다. 노트북 실행을 다시 시작하면 대화 이력은 사라집니다.

실제 프로그램에서는 다음을 결정해야 합니다.

- 사용자별 대화를 어떻게 구분할지
- 대화 내용을 얼마나 오래 저장할지
- 사용자가 대화 기록을 삭제할 수 있는지

In [16]:
chat_history.add_user_message(query_for_nyc) # query_for_nyc에 "뉴욕은?" 추가

answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

뉴욕시는 **2050년 탄소중립**을 목표로 온실가스 감축 정책을 추진하고 있습니다. 특히 건물 부문이 배출의 큰 비중을 차지하기 때문에 건물 에너지 규제가 핵심입니다.

### 1. 대형 건물 온실가스 배출 규제
- 건물별 온실가스 배출 기준을 설정하고, 기준을 초과하면 감축 조치를 요구합니다.
- 건물 소유주가 에너지 효율 개선, 난방 시스템 교체 등을 통해 배출량을 줄이도록 유도합니다.
- 법적 감축 의무를 충족하는 대안으로 **배출권 거래 또는 유사한 시장 기반 제도**를 활용하는 방안도 추진했습니다.
- 이를 통해 법정 기준보다 더 큰 폭의 에너지 절감과 배출 감축을 유도합니다.

### 2. 신축 건물의 제로에너지화
- 최신 에너지 효율 기준을 적용한 성능기반 건축 기준을 도입했습니다.
- 2030년까지 새로 건설되는 건물을 **넷제로 에너지 건물**로 전환하는 것을 목표로 합니다.
- 에너지 손실이 큰 유리 외벽 건물에 대한 추가 규제도 추진하고 있습니다.

### 3. 기존 건물과 공공건물의 에너지 효율 개선
- 건물 단열, 고효율 설비, 에너지 관리시스템 등을 통해 기존 건물의 에너지 사용량을 줄입니다.
- 시 소유 건물에는 에너지 효율 개선과 청정에너지 도입을 우선 적용해 공공부문이 모범을 보이도록 합니다.
- 2017년 기준 시 소유 건물의 온실가스 배출은 2005년보다 약 30% 감소했으며, 1,600개 이상의 공공건물에서 에너지 절감 조치를 시행했습니다.

### 4. 청정전력과 화석연료 대체
- 2050년까지 전력 공급을 **100% 청정에너지**로 전환하는 것을 목표로 합니다.
- 건물의 난방·온수 공급에 사용되는 화석연료 시스템을 고효율 전기 시스템이나 온실가스가 거의 없는 열원으로 바꾸려 합니다.
- 에너지 효율을 높여 건물의 전체 에너지 수요 자체도 줄이는 방식입니다.

### 5. 교통 부문 온실가스 감축
- 뉴욕은 높은 인구밀도와 대중교통 이용률 덕분에 미국 대도시 중 1인당 탄소발자국이 상대적으로 낮습니다.
- 탄소중립을 